In [2]:
import os
import json

In [3]:
models = ['gptoss', 'ollmo7b', 'qwen1_7b']
datasets = ['aime', 'gsm8k', 'math500', 'humaneval', 'kodcode']


root = '/efs/cactts/data'

In [4]:
metrics = ['mean', 'median', 'variance', 'gap', 'entropy', 'gap_probs']
types = ['full', 'tail', 'group_lowest', 'group_highest', 'group_bottom_pct', 'group_top_pct']

# Compute strategies

In [5]:
for model in models:
    for dataset in datasets:
        subdirs = os.listdir(os.path.join(root, model, dataset))
        for sd in subdirs:
            print(f'python recompute_strategies.py /efs/cactts/data/{model}/{dataset}/{sd}/all_response_metrics.jsonl --output /efs/cactts/data/{model}/{dataset}/{sd}/strategy_results_recomputed.jsonl ')

python recompute_strategies.py /efs/cactts/data/gptoss/aime/turn1/all_response_metrics.jsonl --output /efs/cactts/data/gptoss/aime/turn1/strategy_results_recomputed.jsonl 
python recompute_strategies.py /efs/cactts/data/gptoss/aime/turn2/all_response_metrics.jsonl --output /efs/cactts/data/gptoss/aime/turn2/strategy_results_recomputed.jsonl 
python recompute_strategies.py /efs/cactts/data/gptoss/gsm8k/turn3/all_response_metrics.jsonl --output /efs/cactts/data/gptoss/gsm8k/turn3/strategy_results_recomputed.jsonl 
python recompute_strategies.py /efs/cactts/data/gptoss/gsm8k/turn1/all_response_metrics.jsonl --output /efs/cactts/data/gptoss/gsm8k/turn1/strategy_results_recomputed.jsonl 
python recompute_strategies.py /efs/cactts/data/gptoss/gsm8k/turn2/all_response_metrics.jsonl --output /efs/cactts/data/gptoss/gsm8k/turn2/strategy_results_recomputed.jsonl 
python recompute_strategies.py /efs/cactts/data/gptoss/math500/turn1/all_response_metrics.jsonl --output /efs/cactts/data/gptoss/math5

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

def load_all_strategies(root_dir, models, datasets):
    """Load all strategy results from recomputed files."""
    all_data = []
    
    for model in models:
        for dataset in datasets:
            dataset_path = Path(root_dir) / model / dataset
            if not dataset_path.exists():
                continue
            
            # Find all turn directories
            turn_dirs = sorted([d for d in dataset_path.iterdir() if d.is_dir() and d.name.startswith('turn')])
            
            for turn_dir in turn_dirs:
                strats_file = turn_dir / 'strategy_results_recomputed.jsonl'
                if strats_file.exists():
                    with open(strats_file, 'r') as f:
                        data = json.load(f)
                    
                    turn_num = turn_dir.name
                    strategy_results = data.get('strategy_results', {})
                    
                    for strategy_name, stats in strategy_results.items():
                        all_data.append({
                            'model': model,
                            'dataset': dataset,
                            'turn': turn_num,
                            'strategy': strategy_name,
                            'correct': stats.get('correct', 0),
                            'total': stats.get('total', 0),
                            'accuracy': stats.get('accuracy', 0.0)
                        })
    
    return pd.DataFrame(all_data)

# Load all data
print("Loading all strategy results...")
df = load_all_strategies(root, models, datasets)
print(f"Loaded {len(df)} records")
print(f"Models: {df['model'].nunique()}, Datasets: {df['dataset'].nunique()}, Strategies: {df['strategy'].nunique()}")
df.head(10)

Loading all strategy results...
Loaded 2738 records
Models: 3, Datasets: 5, Strategies: 74


,model,dataset,turn,strategy,correct,total,accuracy
0,gptoss,aime,turn1,random,28,30,0.933333
1,gptoss,aime,turn1,oracle,29,30,0.966667
2,gptoss,aime,turn1,highest_mean_full,27,30,0.900000
3,gptoss,aime,turn1,lowest_mean_full,28,30,0.933333
4,gptoss,aime,turn1,highest_mean_tail,26,30,0.866667
5,gptoss,aime,turn1,lowest_mean_tail,27,30,0.900000
6,gptoss,aime,turn1,highest_mean_group_lowest,28,30,0.933333
7,gptoss,aime,turn1,lowest_mean_group_lowest,27,30,0.900000
8,gptoss,aime,turn1,highest_mean_group_highest,27,30,0.900000
9,gptoss,aime,turn1,lowest_mean_group_highest,25,30,0.833333


In [7]:
def create_averaged_table(df, dataset_name):
    """Create table for a specific dataset, averaged across turns."""
    dataset_df = df[df['dataset'] == dataset_name].copy()
    
    # Group by model and strategy, average across turns
    avg_df = dataset_df.groupby(['model', 'strategy']).agg({
        'correct': 'sum',
        'total': 'sum',
        'accuracy': 'mean'
    }).reset_index()
    
    # Recalculate accuracy from summed correct/total
    avg_df['accuracy'] = avg_df['correct'] / avg_df['total']
    
    # Pivot to have strategies as rows, models as columns
    pivot_table = avg_df.pivot(index='strategy', columns='model', values='accuracy')
    
    # Sort by average accuracy across models
    pivot_table['average'] = pivot_table.mean(axis=1)
    pivot_table = pivot_table.sort_values('average', ascending=False)
    
    return pivot_table

# Create tables for each dataset
for dataset in datasets:
    dataset_data = df[df['dataset'] == dataset]
    if len(dataset_data) > 0:
        print(f"\n{'='*80}")
        print(f"Dataset: {dataset.upper()}")
        print(f"{'='*80}")
        table = create_averaged_table(df, dataset)
        print(table.round(4))
        print(f"\nTop 5 strategies:")
        print(table.head().round(4))


Dataset: AIME
model                              gptoss  ollmo7b  qwen1_7b  average
strategy                                                             
oracle                             0.9667   0.0111    0.6889   0.5556
highest_mean_group_lowest          0.9000   0.0000    0.5000   0.4667
highest_median_group_lowest        0.9000   0.0000    0.4667   0.4556
highest_variance_group_bottom_pct  0.8500   0.0000    0.5111   0.4537
highest_mean_group_bottom_pct      0.8500   0.0000    0.4889   0.4463
...                                   ...      ...       ...      ...
lowest_entropy_group_bottom_pct    0.8833   0.0000    0.2778   0.3870
lowest_gap_tail                    0.8667   0.0000    0.2333   0.3667
lowest_mean_tail                   0.8667   0.0000    0.2000   0.3556
lowest_median_tail                 0.8667   0.0000    0.2000   0.3556
lowest_variance_tail               0.8500   0.0000    0.2000   0.3500

[74 rows x 4 columns]

Top 5 strategies:
model                            

In [8]:
# Alternative: Create one comprehensive table per dataset with more details
def create_detailed_table(df, dataset_name):
    """Create detailed table showing strategy performance per model."""
    dataset_df = df[df['dataset'] == dataset_name].copy()
    
    # Group by model and strategy
    summary = dataset_df.groupby(['model', 'strategy']).agg({
        'correct': 'sum',
        'total': 'sum',
        'accuracy': 'mean'
    }).reset_index()
    
    summary['accuracy_from_sum'] = summary['correct'] / summary['total']
    
    return summary.sort_values(['model', 'accuracy_from_sum'], ascending=[True, False])

# Export to CSV for each dataset
output_dir = Path('strategy_tables')
output_dir.mkdir(exist_ok=True)

for dataset in datasets:
    dataset_data = df[df['dataset'] == dataset]
    if len(dataset_data) > 0:
        # Pivot table
        table = create_averaged_table(df, dataset)
        table.to_csv(output_dir / f'{dataset}_strategies_averaged.csv')
        
        # Detailed table
        detailed = create_detailed_table(df, dataset)
        detailed.to_csv(output_dir / f'{dataset}_strategies_detailed.csv', index=False)
        
        print(f"Saved tables for {dataset}")

print(f"\nAll tables saved to {output_dir}/")

Saved tables for aime
Saved tables for gsm8k
Saved tables for math500
Saved tables for humaneval
Saved tables for kodcode

All tables saved to strategy_tables/


# Summary Statistics

View top strategies across all datasets and models

In [9]:
# Overall best strategies across all datasets and models
overall_summary = df.groupby('strategy').agg({
    'correct': 'sum',
    'total': 'sum',
    'accuracy': 'mean'
}).reset_index()

overall_summary['accuracy_from_sum'] = overall_summary['correct'] / overall_summary['total']
overall_summary = overall_summary.sort_values('accuracy_from_sum', ascending=False)

print("Top 20 Strategies Overall (averaged across all models and datasets):")
print(overall_summary.head(20)[['strategy', 'accuracy_from_sum', 'correct', 'total']].to_string(index=False))

print("\n\nBottom 10 Strategies:")
print(overall_summary.tail(10)[['strategy', 'accuracy_from_sum', 'correct', 'total']].to_string(index=False))

Top 20 Strategies Overall (averaged across all models and datasets):
                       strategy  accuracy_from_sum  correct  total
                         oracle           0.858863    15767  18358
              highest_mean_tail           0.570433    10472  18358
               highest_gap_tail           0.569779    10460  18358
lowest_entropy_group_bottom_pct           0.569452    10454  18358
            highest_median_tail           0.569125    10448  18358
      highest_gap_group_top_pct           0.568526    10437  18358
              highest_mean_full           0.567709    10422  18358
     highest_mean_group_top_pct           0.567600    10420  18358
               highest_gap_full           0.567545    10419  18358
          highest_variance_tail           0.567491    10418  18358
   highest_median_group_top_pct           0.567273    10414  18358
            highest_median_full           0.566837    10406  18358
     highest_mean_group_highest           0.566619    10402 

# Structured Analysis by Metric Components

Parse strategy names and analyze by metric, type, and selection criteria

In [10]:
def parse_strategy_name(strategy_name):
    """Parse strategy name into components."""
    if strategy_name in ['random', 'oracle']:
        return {
            'selection': strategy_name,
            'metric': strategy_name,
            'type': strategy_name
        }
    
    # Extract selection criteria (highest/lowest)
    if strategy_name.startswith('highest_'):
        selection = 'highest'
        remainder = strategy_name[8:]  # Remove 'highest_'
    elif strategy_name.startswith('lowest_'):
        selection = 'lowest'
        remainder = strategy_name[7:]  # Remove 'lowest_'
    else:
        return None
    
    # Parse metric and type
    # Possible metrics: mean, median, variance, gap, entropy, gap_probs
    # Possible types: full, tail, group_lowest, group_highest, group_bottom_pct, group_top_pct
    
    for metric in ['mean', 'median', 'variance', 'gap_probs', 'gap', 'entropy']:
        if remainder.startswith(metric + '_'):
            type_part = remainder[len(metric) + 1:]
            return {
                'selection': selection,
                'metric': metric,
                'type': type_part
            }
    
    return None

# Parse all strategies
df['parsed'] = df['strategy'].apply(parse_strategy_name)
df_parsed = df[df['parsed'].notna()].copy()
df_parsed['selection'] = df_parsed['parsed'].apply(lambda x: x['selection'])
df_parsed['metric'] = df_parsed['parsed'].apply(lambda x: x['metric'])
df_parsed['type'] = df_parsed['parsed'].apply(lambda x: x['type'])

print(f"Successfully parsed {len(df_parsed)} out of {len(df)} records")
df_parsed.head(10)

Successfully parsed 2738 out of 2738 records


,model,dataset,turn,strategy,correct,total,accuracy,parsed,selection,metric,type
0,gptoss,aime,turn1,random,28,30,0.933333,"{'selection': 'random', 'metric': 'random', 't...",random,random,random
1,gptoss,aime,turn1,oracle,29,30,0.966667,"{'selection': 'oracle', 'metric': 'oracle', 't...",oracle,oracle,oracle
2,gptoss,aime,turn1,highest_mean_full,27,30,0.900000,"{'selection': 'highest', 'metric': 'mean', 'ty...",highest,mean,full
3,gptoss,aime,turn1,lowest_mean_full,28,30,0.933333,"{'selection': 'lowest', 'metric': 'mean', 'typ...",lowest,mean,full
4,gptoss,aime,turn1,highest_mean_tail,26,30,0.866667,"{'selection': 'highest', 'metric': 'mean', 'ty...",highest,mean,tail
5,gptoss,aime,turn1,lowest_mean_tail,27,30,0.900000,"{'selection': 'lowest', 'metric': 'mean', 'typ...",lowest,mean,tail
6,gptoss,aime,turn1,highest_mean_group_lowest,28,30,0.933333,"{'selection': 'highest', 'metric': 'mean', 'ty...",highest,mean,group_lowest
7,gptoss,aime,turn1,lowest_mean_group_lowest,27,30,0.900000,"{'selection': 'lowest', 'metric': 'mean', 'typ...",lowest,mean,group_lowest
8,gptoss,aime,turn1,highest_mean_group_highest,27,30,0.900000,"{'selection': 'highest', 'metric': 'mean', 'ty...",highest,mean,group_highest
9,gptoss,aime,turn1,lowest_mean_group_highest,25,30,0.833333,"{'selection': 'lowest', 'metric': 'mean', 'typ...",lowest,mean,group_highest


In [11]:
# Filter out random and oracle for metric-based analysis
df_metrics = df_parsed[~df_parsed['strategy'].isin(['random', 'oracle'])].copy()

# Compute mean and std of accuracy across all models, datasets, and turns
structured_summary = df_metrics.groupby(['metric', 'type', 'selection'])['accuracy'].agg(['mean', 'std', 'count']).reset_index()

# Round to 3 decimal places
structured_summary['mean'] = structured_summary['mean'].round(3)
structured_summary['std'] = structured_summary['std'].round(3)

# Rename columns for clarity
structured_summary.columns = ['Metric', 'Aggregation Type', 'Selection', 'Accuracy', 'Std', 'Count']

# Sort by accuracy descending
structured_summary = structured_summary.sort_values('Accuracy', ascending=False)

print("Structured Summary of All Strategies (Mean ± Std across all models, datasets, and turns):")
print(f"\nTotal configurations: {len(structured_summary)}")
print(f"\n{structured_summary.to_string(index=False)}")

# Save to CSV
structured_summary.to_csv(output_dir / 'structured_strategy_summary.csv', index=False)
print(f"\nSaved to {output_dir / 'structured_strategy_summary.csv'}")

Structured Summary of All Strategies (Mean ± Std across all models, datasets, and turns):

Total configurations: 72

   Metric Aggregation Type Selection  Accuracy   Std  Count
     mean             tail   highest     0.524 0.288     37
      gap             tail   highest     0.523 0.289     37
   median             tail   highest     0.523 0.288     37
 variance             full   highest     0.522 0.287     37
 variance             tail   highest     0.521 0.289     37
 variance group_bottom_pct   highest     0.521 0.285     37
     mean             full   highest     0.521 0.287     37
   median group_bottom_pct   highest     0.521 0.286     37
     mean group_bottom_pct   highest     0.520 0.287     37
   median             full   highest     0.520 0.287     37
      gap     group_lowest   highest     0.517 0.280     37
      gap             full   highest     0.516 0.290     37
     mean    group_top_pct   highest     0.515 0.292     37
      gap    group_top_pct   highest     0.

In [12]:
# Create separate views for highest and lowest
print("\n" + "="*100)
print("TOP 20 - HIGHEST Selection Strategies:")
print("="*100)
highest_only = structured_summary[structured_summary['Selection'] == 'highest'].head(20)
print(highest_only.to_string(index=False))

print("\n" + "="*100)
print("TOP 20 - LOWEST Selection Strategies:")
print("="*100)
lowest_only = structured_summary[structured_summary['Selection'] == 'lowest'].head(20)
print(lowest_only.to_string(index=False))


TOP 20 - HIGHEST Selection Strategies:
   Metric Aggregation Type Selection  Accuracy   Std  Count
     mean             tail   highest     0.524 0.288     37
      gap             tail   highest     0.523 0.289     37
   median             tail   highest     0.523 0.288     37
 variance             full   highest     0.522 0.287     37
 variance             tail   highest     0.521 0.289     37
 variance group_bottom_pct   highest     0.521 0.285     37
     mean             full   highest     0.521 0.287     37
   median group_bottom_pct   highest     0.521 0.286     37
     mean group_bottom_pct   highest     0.520 0.287     37
   median             full   highest     0.520 0.287     37
      gap     group_lowest   highest     0.517 0.280     37
      gap             full   highest     0.516 0.290     37
     mean    group_top_pct   highest     0.515 0.292     37
      gap    group_top_pct   highest     0.514 0.291     37
 variance    group_top_pct   highest     0.514 0.285     37


In [13]:
# Group by metric and type, showing best selection for each
best_by_metric_type = df_metrics.groupby(['metric', 'type', 'selection'])['accuracy'].mean().reset_index()
best_by_metric_type = best_by_metric_type.sort_values('accuracy', ascending=False).groupby(['metric', 'type']).first().reset_index()

# Add std
std_by_metric_type = df_metrics.groupby(['metric', 'type', 'selection'])['accuracy'].std().reset_index()
std_by_metric_type.columns = ['metric', 'type', 'selection', 'std']

best_combined = best_by_metric_type.merge(std_by_metric_type, on=['metric', 'type', 'selection'], how='left')
best_combined['accuracy'] = best_combined['accuracy'].round(3)
best_combined['std'] = best_combined['std'].round(3)
best_combined.columns = ['Metric', 'Aggregation Type', 'Best Selection', 'Accuracy', 'Std']

print("\n" + "="*100)
print("BEST Strategy per Metric-Type Combination:")
print("="*100)
print(best_combined.sort_values('Accuracy', ascending=False).to_string(index=False))

best_combined.to_csv(output_dir / 'best_per_metric_type.csv', index=False)


BEST Strategy per Metric-Type Combination:
   Metric Aggregation Type Best Selection  Accuracy   Std
     mean             tail        highest     0.524 0.288
      gap             tail        highest     0.523 0.289
   median             tail        highest     0.523 0.288
 variance             full        highest     0.522 0.287
   median group_bottom_pct        highest     0.521 0.286
     mean             full        highest     0.521 0.287
 variance group_bottom_pct        highest     0.521 0.285
 variance             tail        highest     0.521 0.289
     mean group_bottom_pct        highest     0.520 0.287
   median             full        highest     0.520 0.287
      gap     group_lowest        highest     0.517 0.280
      gap             full        highest     0.516 0.290
     mean    group_top_pct        highest     0.515 0.292
      gap    group_top_pct        highest     0.514 0.291
 variance    group_top_pct        highest     0.514 0.285
   median    group_top_pct  

# Dataset-wise Strategy Performance

Each row is a strategy, each column is a dataset (not averaged across datasets)

In [14]:
# Create dataset-wise table with all strategies including oracle and random
# Average across models and turns, but keep datasets separate

# Filter to only include desired aggregation types
desired_types = ['full', 'tail', 'group_bottom_pct', 'group_top_pct']
df_filtered = df_parsed[
    df_parsed['type'].isin(desired_types) | df_parsed['metric'].isin(['oracle', 'random'])
].copy()

# Compute mean for each dataset
dataset_strategy_mean = df_filtered.groupby(['dataset', 'metric', 'type', 'selection'])['accuracy'].mean().reset_index()
pivot_mean = dataset_strategy_mean.pivot_table(
    index=['metric', 'type', 'selection'],
    columns='dataset',
    values='accuracy'
).reset_index()

# Compute std for each dataset
dataset_strategy_std = df_filtered.groupby(['dataset', 'metric', 'type', 'selection'])['accuracy'].std().reset_index()
pivot_std = dataset_strategy_std.pivot_table(
    index=['metric', 'type', 'selection'],
    columns='dataset',
    values='accuracy'
).reset_index()

# Get all dataset columns (sorted alphabetically)
dataset_cols = sorted([col for col in pivot_mean.columns if col not in ['metric', 'type', 'selection']])

# Merge mean and std, interleaving columns
pivot_by_dataset = pivot_mean[['metric', 'type', 'selection']].copy()

for dataset in dataset_cols:
    # Add mean column
    pivot_by_dataset[dataset] = pivot_mean[dataset]
    # Add std column
    pivot_by_dataset[f'{dataset}_std'] = pivot_std[dataset]

# Round all values to 3 decimals
for col in pivot_by_dataset.columns:
    if col not in ['metric', 'type', 'selection']:
        pivot_by_dataset[col] = pivot_by_dataset[col].round(3)

# Rename columns for clarity
pivot_by_dataset = pivot_by_dataset.rename(columns={
    'metric': 'Metric',
    'type': 'Aggregation Type',
    'selection': 'Selection'
})

# Create sorting keys
# 1. Oracle and random first
pivot_by_dataset['sort_priority'] = pivot_by_dataset['Metric'].apply(
    lambda x: 0 if x in ['oracle', 'random'] else 1
)

# 2. Define metric order
metric_order = {'mean': 0, 'variance': 1, 'median': 2, 'gap': 3, 'entropy': 4, 'gap_probs': 5, 'oracle': 6, 'random': 7}
pivot_by_dataset['metric_order'] = pivot_by_dataset['Metric'].map(metric_order).fillna(99)

# 3. Define type order
type_order = {'full': 0, 'tail': 1, 'group_bottom_pct': 2, 'group_top_pct': 3, 'oracle': 4, 'random': 5}
pivot_by_dataset['type_order'] = pivot_by_dataset['Aggregation Type'].map(type_order).fillna(99)

# Sort: oracle/random first, then by metric order, type order, selection
pivot_by_dataset = pivot_by_dataset.sort_values(
    ['sort_priority', 'metric_order', 'type_order', 'Selection'],
    ascending=[True, True, True, True]
)

# Drop the sort columns
pivot_by_dataset = pivot_by_dataset.drop(columns=['sort_priority', 'metric_order', 'type_order'])

print("Strategy Performance by Dataset (averaged across models and turns):")
print(f"\n{pivot_by_dataset.to_string(index=False)}")

# Save to CSV
pivot_by_dataset.to_csv(output_dir / 'strategies_by_dataset_full.csv', index=False)
print(f"\nSaved to {output_dir / 'strategies_by_dataset_full.csv'}")

Strategy Performance by Dataset (averaged across models and turns):

   Metric Aggregation Type Selection  aime  aime_std  gsm8k  gsm8k_std  humaneval  humaneval_std  kodcode  kodcode_std  math500  math500_std
   oracle           oracle    oracle 0.504     0.425  0.963      0.041      0.834          0.098    0.681        0.326    0.856        0.144
   random           random    random 0.371     0.383  0.597      0.193      0.491          0.182    0.373        0.233    0.682        0.308
     mean             full   highest 0.379     0.358  0.649      0.212      0.505          0.189    0.367        0.234    0.699        0.284
     mean             full    lowest 0.325     0.393  0.484      0.198      0.453          0.194    0.329        0.202    0.640        0.326
     mean             tail   highest 0.388     0.364  0.649      0.212      0.506          0.188    0.374        0.242    0.700        0.285
     mean             tail    lowest 0.292     0.367  0.487      0.202      0.438    

In [15]:
# Show oracle and random first, then top strategies
print("\n" + "="*120)
print("ORACLE AND RANDOM PERFORMANCE:")
print("="*120)
oracle_random = pivot_by_dataset[pivot_by_dataset['Metric'].isin(['oracle', 'random'])]
print(oracle_random.to_string(index=False))

print("\n" + "="*120)
print("TOP 20 METRIC-BASED STRATEGIES:")
print("="*120)
non_baseline = pivot_by_dataset[~pivot_by_dataset['Metric'].isin(['oracle', 'random'])]
top_20 = non_baseline.head(20)
print(top_20.to_string(index=False))


ORACLE AND RANDOM PERFORMANCE:
Metric Aggregation Type Selection  aime  aime_std  gsm8k  gsm8k_std  humaneval  humaneval_std  kodcode  kodcode_std  math500  math500_std
oracle           oracle    oracle 0.504     0.425  0.963      0.041      0.834          0.098    0.681        0.326    0.856        0.144
random           random    random 0.371     0.383  0.597      0.193      0.491          0.182    0.373        0.233    0.682        0.308

TOP 20 METRIC-BASED STRATEGIES:
  Metric Aggregation Type Selection  aime  aime_std  gsm8k  gsm8k_std  humaneval  humaneval_std  kodcode  kodcode_std  math500  math500_std
    mean             full   highest 0.379     0.358  0.649      0.212      0.505          0.189    0.367        0.234    0.699        0.284
    mean             full    lowest 0.325     0.393  0.484      0.198      0.453          0.194    0.329        0.202    0.640        0.326
    mean             tail   highest 0.388     0.364  0.649      0.212      0.506          0.188    0.

In [16]:
# Create per-dataset analysis showing top strategies for each dataset
for dataset in datasets:
    if dataset in pivot_by_dataset.columns:
        print(f"\n{'='*80}")
        print(f"TOP 10 STRATEGIES FOR {dataset.upper()}:")
        print(f"{'='*80}")
        dataset_sorted = pivot_by_dataset.sort_values(dataset, ascending=False)
        top_10 = dataset_sorted.head(10)[['Metric', 'Aggregation Type', 'Selection', dataset]]
        print(top_10.to_string(index=False))


TOP 10 STRATEGIES FOR AIME:
  Metric Aggregation Type Selection  aime
  oracle           oracle    oracle 0.504
variance group_bottom_pct   highest 0.404
    mean group_bottom_pct   highest 0.396
  median group_bottom_pct   highest 0.396
variance             full   highest 0.392
variance             tail   highest 0.388
  median             tail   highest 0.388
    mean             tail   highest 0.388
    mean             full   highest 0.379
  median             full   highest 0.379

TOP 10 STRATEGIES FOR GSM8K:
   Metric Aggregation Type Selection  gsm8k
   oracle           oracle    oracle  0.963
  entropy group_bottom_pct    lowest  0.672
gap_probs    group_top_pct   highest  0.660
      gap    group_top_pct   highest  0.658
     mean    group_top_pct   highest  0.656
   median    group_top_pct   highest  0.654
     mean             tail   highest  0.649
     mean             full   highest  0.649
   median             full   highest  0.649
   median             tail   highest  0

# Per-Model Strategy Tables

Create separate tables for each model to reduce variance

In [ ]:
def create_model_table(df_parsed, model_name, datasets, output_dir):
    """Create strategy table for a specific model."""
    
    # Filter to desired aggregation types and specific model
    desired_types = ['full', 'tail', 'group_bottom_pct', 'group_top_pct']
    df_model = df_parsed[
        (df_parsed['model'] == model_name) &
        (df_parsed['type'].isin(desired_types) | df_parsed['metric'].isin(['oracle', 'random']))
    ].copy()
    
    if len(df_model) == 0:
        print(f"Warning: No data found for model {model_name}")
        return None
    
    # Compute mean for each dataset
    dataset_strategy_mean = df_model.groupby(['dataset', 'metric', 'type', 'selection'])['accuracy'].mean().reset_index()
    pivot_mean = dataset_strategy_mean.pivot_table(
        index=['metric', 'type', 'selection'],
        columns='dataset',
        values='accuracy'
    ).reset_index()
    
    # Compute std for each dataset
    dataset_strategy_std = df_model.groupby(['dataset', 'metric', 'type', 'selection'])['accuracy'].std().reset_index()
    pivot_std = dataset_strategy_std.pivot_table(
        index=['metric', 'type', 'selection'],
        columns='dataset',
        values='accuracy'
    ).reset_index()
    
    # Get all dataset columns that actually exist in the data (sorted alphabetically)
    dataset_cols = sorted([col for col in pivot_mean.columns if col not in ['metric', 'type', 'selection']])
    
    # Merge mean and std, interleaving columns
    pivot_by_dataset = pivot_mean[['metric', 'type', 'selection']].copy()
    
    for dataset in dataset_cols:
        # Add mean column (only if it exists)
        if dataset in pivot_mean.columns:
            pivot_by_dataset[dataset] = pivot_mean[dataset]
        # Add std column (only if it exists)
        if dataset in pivot_std.columns:
            pivot_by_dataset[f'{dataset}_std'] = pivot_std[dataset]
    
    # Round all values to 3 decimals
    for col in pivot_by_dataset.columns:
        if col not in ['metric', 'type', 'selection']:
            pivot_by_dataset[col] = pivot_by_dataset[col].round(3)
    
    # Rename columns for clarity
    pivot_by_dataset = pivot_by_dataset.rename(columns={
        'metric': 'Metric',
        'type': 'Aggregation Type',
        'selection': 'Selection'
    })
    
    # Create sorting keys
    pivot_by_dataset['sort_priority'] = pivot_by_dataset['Metric'].apply(
        lambda x: 0 if x in ['oracle', 'random'] else 1
    )
    
    # Define metric order
    metric_order = {'mean': 0, 'variance': 1, 'median': 2, 'gap': 3, 'entropy': 4, 'gap_probs': 5, 'oracle': 6, 'random': 7}
    pivot_by_dataset['metric_order'] = pivot_by_dataset['Metric'].map(metric_order).fillna(99)
    
    # Define type order
    type_order = {'full': 0, 'tail': 1, 'group_bottom_pct': 2, 'group_top_pct': 3, 'oracle': 4, 'random': 5}
    pivot_by_dataset['type_order'] = pivot_by_dataset['Aggregation Type'].map(type_order).fillna(99)
    
    # Sort
    pivot_by_dataset = pivot_by_dataset.sort_values(
        ['sort_priority', 'metric_order', 'type_order', 'Selection'],
        ascending=[True, True, True, True]
    )
    
    # Drop the sort columns
    pivot_by_dataset = pivot_by_dataset.drop(columns=['sort_priority', 'metric_order', 'type_order'])
    
    return pivot_by_dataset

# Create tables for each model
for model in models:
    print(f"\n{'='*120}")
    print(f"MODEL: {model.upper()}")
    print(f"{'='*120}")
    
    model_table = create_model_table(df_parsed, model, datasets, output_dir)
    
    if model_table is not None:
        print(model_table.to_string(index=False))
        
        # Save to CSV
        output_file = output_dir / f'strategies_{model}.csv'
        model_table.to_csv(output_file, index=False)
        print(f"\nSaved to {output_file}")
    else:
        print(f"Skipping {model} - no data available")


MODEL: GPTOSS
   Metric Aggregation Type Selection  aime  aime_std  gsm8k  gsm8k_std  humaneval  humaneval_std  kodcode  kodcode_std  math500  math500_std
   oracle           oracle    oracle 0.967     0.000  0.975      0.002      0.940          0.003    0.938        0.001    0.977        0.001
   random           random    random 0.933     0.000  0.548      0.016      0.691          0.042    0.601        0.002    0.947        0.013
     mean             full   highest 0.850     0.071  0.455      0.009      0.729          0.007    0.596        0.020    0.943        0.004
     mean             full    lowest 0.933     0.000  0.676      0.018      0.648          0.020    0.512        0.008    0.946        0.008
     mean             tail   highest 0.867     0.000  0.455      0.008      0.721          0.045    0.617        0.019    0.946        0.009
     mean             tail    lowest 0.867     0.047  0.683      0.017      0.619          0.003    0.508        0.004    0.938        0.00

: 